# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and analyzing the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In [ ]:
# Install `mlcroissant` if not already installed
!pip install mlcroissant

## 1. Data Loading
Load schema and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# View main metadata attributes
print(f"Dataset Title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Version: {dataset.metadata.version}")

## 2. Data Overview
Review available record sets and their fields using their `@id` attributes. All entities are referenced by their `@id` for consistency and reproducibility.

In [ ]:
# List all available record sets by @id
record_sets = list(dataset.record_sets)
print("Available record set @id values:")
for record_set in record_sets:
    print(f"- @id: {record_set['@id']}")
    print(f"  Name: {record_set.get('name', '')}")
    # List fields and their @ids
    fields = record_set.get('field', [])
    print(f"  Fields ({len(fields)}):")
    for field in fields:
        if isinstance(field, dict):
            field_id = field.get('@id', 'N/A')
            field_name = field.get('name', '')
        else:
            field_id = field
            field_name = ''
        print(f"    - @id: {field_id}  name: {field_name}")
    print()

## 3. Data Extraction
Load data from each available record set into a DataFrame. All addressing uses record set and field `@id` values as discovered above.

In [ ]:
# Gather all record set @id values for iteration
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

# Load all dataframes (may take time depending on record size and availability)
for record_set_id in record_set_ids:
    rows = list(dataset.records(record_set=record_set_id))
    print(f"Loaded {len(rows)} rows for record set {record_set_id}")
    if rows:
        dataframes[record_set_id] = pd.DataFrame(rows)
    else:
        print(f"Warning: No rows found for {record_set_id}")

# Select main record set for demonstration (replace with the prominent one if known)
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id and main_record_set_id in dataframes:
    print(f"Available columns in main record set ({main_record_set_id}):")
    print(dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply typical data processing operations: filter records, normalize a numeric field, and group by a categorical field. Use field `@id`s wherever necessary.

In [ ]:
# Choose a numeric field for EDA. Replace with the actual field @id found earlier if needed.
# Here, as an example, we'll try with 'coverage_percentage', which is a common mass spec metric.
numeric_field_id = None
if main_record_set_id and main_record_set_id in dataframes:
    # Try to auto-select a numeric field
    for col in dataframes[main_record_set_id].columns:
        if 'coverage' in col.lower() or 'count' in col.lower() or 'abundance' in col.lower() or 'mw' in col.lower():
            numeric_field_id = col
            break
    if not numeric_field_id:
        print("No obvious numeric field found, please select one based on the column list above.")
else:
    print("No main record set loaded.")

if numeric_field_id:
    df = dataframes[main_record_set_id]
    # Convert numeric field to float if possible
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

    # Filtering: keep only values greater than threshold
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold} (count: {len(filtered_df)}):")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean())/filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by: try to find a categorical field (e.g., 'description', 'protein_name', ...)
    group_field_id = None
    for col in df.columns:
        if 'desc' in col.lower() or 'name' in col.lower() or 'sample' in col.lower():
            group_field_id = col
            break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nData grouped by {group_field_id} (mean of {numeric_field_id}):")
        print(grouped_df.head())
    else:
        print("No suitable group-by field found.")
else:
    print("Please re-run after identifying a numeric field such as 'coverage_percentage', 'count', or 'abundance'.")

## 5. Visualization
Visualize the distribution of the numeric field and/or the group means. Adjust field @ids as appropriate.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and main_record_set_id and main_record_set_id in dataframes:
    plt.figure(figsize=(8,4))
    sns.histplot(dataframes[main_record_set_id][numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id} in {main_record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id:
        plt.figure(figsize=(10,4))
        grouped_df = dataframes[main_record_set_id].groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False)
        grouped_df.head(30).plot(kind="bar")
        plt.title(f"Mean {numeric_field_id} by {group_field_id} (Top 30)")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR^2 Mass Spectrometry dataset using its Croissant schema and explored its structure using only `@id` references for all entities. After loading data into DataFrames, we filtered, normalized, and grouped numeric features, and visualized distributions to gain first insights. For in-depth studies, consult the "fields" and metadata to perform domain-specific analysis or integrate with bioinformatics pipelines.

*This notebook was generated to provide a reproducible, standards-based data exploration using the mlcroissant library and the FAIR^2 Croissant metadata schema.*